# Path marking versus phase programming: four-path witness and exploratory survey

This notebook reproduces every numerical result and the figure of the article *Path Marking versus Phase Programming for Programmable Optical Switching: Dominance, Classical Emulability and a Four-Path Witness*.

**Model.** For $M$ paths with fixed amplitudes, ideal path marking produces a coherence matrix $G$ that is positive semidefinite with unit diagonal (set $\mathcal{E}_M$). Phase programming, including any classical random-phase ensemble, produces the convex hull $\mathcal{P}_M$ of rank-one matrices $vv^\dagger$ with $|v_j|=1$.

**Contents.**
1. Four-path witness: the projector $P$ onto the range of the rank-two marking matrix $G_0$ (Jones vectors H, V, D, circular). The collected fraction $\mathrm{tr}(PG)/4$ is 1 for $G_0$, and its phase-only maximum $c^\star$ is evaluated in closed form and confirmed by branch and bound.
2. Far-field readout of $G_0$ from four apertures with distinct baselines.
3. Robustness under path-phase randomisation, and the figure.
4. Exploratory survey for $M=2,\dots,5$ (semidefinite programming versus multi-start phase optimisation).

All results are written to `results/`. Runtime: about a minute for parts 1–3 and several minutes for part 4.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "cvxpy", "matplotlib"], check=True)
import json, os, time, numpy as np, cvxpy as cp, matplotlib
import matplotlib.pyplot as plt
from scipy.optimize import minimize
os.makedirs("results", exist_ok=True)
print("numpy", np.__version__, "| cvxpy", cp.__version__)

numpy 2.5.3 | cvxpy 1.9.3


## 1. Four-path witness and closed-form phase-only maximum

In [2]:
s = np.sqrt(2)
V = np.array([[1, 0], [0, 1], [1/s, 1/s], [1/s, 1j/s]])      # Jones vectors (rows)
G0 = V @ V.conj().T                                          # marking matrix, rank 2
P = V @ np.linalg.inv(V.conj().T @ V) @ V.conj().T           # projector onto range(G0)
M = 4
print("tr(P G0)/4 =", np.real(np.trace(P @ G0))/4)

def f(th):
    v = np.concatenate([np.ones((len(th), 1)), np.exp(1j*th)], axis=1)
    return np.real(np.einsum("ni,ij,nj->n", v.conj(), P, v))
def grad(th):
    v = np.concatenate([np.ones((len(th), 1)), np.exp(1j*th)], axis=1)
    return (2*np.imag(v.conj()*(v @ P.T)))[:, 1:]

th_star = np.pi*np.array([[7/4, 15/8, 1/8]])
c_star = 0.5 + ((4 - s)*np.sin(np.pi/8) + (2 + 3*s)*np.cos(np.pi/8))/14
print("closed form c* =", c_star, "| value at maximiser =", f(th_star)[0]/4)
assert abs(f(th_star)[0]/4 - c_star) < 1e-14

tr(P G0)/4 = 0.9999999999999998
closed form c* = 0.9826418277832761 | value at maximiser = 0.9826418277832758


Branch and bound over the three relative phases. Each box gets the second-order upper bound $f(c)+h\|\nabla f(c)\|_1+\tfrac12 h^2\sum_{ij}B_{ij}$, with Hessian bounds $B_{ij}=2|P_{ij}|$ ($i\ne j$) and $B_{jj}=2\sum_k|P_{jk}|$. Boxes that cannot exceed the best value by more than $10^{-6}$ are discarded. Double precision, no directed rounding.

In [3]:
Hb = 2*np.abs(P[1:, 1:]).copy()
np.fill_diagonal(Hb, [2*np.abs(P[j]).sum() for j in range(1, M)])
best, tol, n0 = f(th_star)[0], 1e-6, 32
h = np.pi/n0; g = (np.arange(n0) + 0.5)*2*h
Cx = np.stack(np.meshgrid(g, g, g, indexing="ij"), -1).reshape(-1, 3); level = 0; t0 = time.time()
while len(Cx):
    fc = f(Cx); best = max(best, fc.max())
    ub = fc + np.abs(grad(Cx)).sum(1)*h + 0.5*h*h*Hb.sum()
    Cx = Cx[ub > best + tol]
    print(f"level {level}: half-width {h:.2e}, boxes kept {len(Cx)}, elapsed {time.time()-t0:.1f}s", flush=True)
    if not len(Cx): break
    h /= 2; off = np.array(np.meshgrid([-h, h], [-h, h], [-h, h], indexing="ij")).reshape(3, -1).T
    Cx = (Cx[:, None, :] + off[None]).reshape(-1, 3); level += 1
cert_ub = (best + tol)/4
print(f"phase-only maximum: c* = {c_star:.9f} <= max <= {cert_ub:.9f}")
print(f"margin 1 - c* = {1-c_star:.4%}")

level 0: half-width 9.82e-02, boxes kept 192, elapsed 0.0s
level 1: half-width 4.91e-02, boxes kept 202, elapsed 0.0s
level 2: half-width 2.45e-02, boxes kept 204, elapsed 0.0s
level 3: half-width 1.23e-02, boxes kept 202, elapsed 0.0s
level 4: half-width 6.14e-03, boxes kept 202, elapsed 0.0s
level 5: half-width 3.07e-03, boxes kept 202, elapsed 0.0s
level 6: half-width 1.53e-03, boxes kept 200, elapsed 0.0s
level 7: half-width 7.67e-04, boxes kept 176, elapsed 0.0s
level 8: half-width 3.83e-04, boxes kept 18, elapsed 0.0s
level 9: half-width 1.92e-04, boxes kept 0, elapsed 0.0s
phase-only maximum: c* = 0.982641828 <= max <= 0.982642078
margin 1 - c* = 1.7358%


## 2. Far-field readout

In [4]:
r = np.array([[0, 0], [1, 0], [0, 3], [4, 2]])
diffs = [tuple(r[j]-r[l]) for j in range(M) for l in range(M) if j != l]
print("pairwise difference vectors distinct:", len(set(diffs)) == len(diffs))
k = np.linspace(-np.pi, np.pi, 128, endpoint=False); KX, KY = np.meshgrid(k, k)
ph = lambda j, l: np.exp(-1j*(KX*(r[j, 0]-r[l, 0]) + KY*(r[j, 1]-r[l, 1])))
I = sum(0.25*G0[j, l]*ph(j, l) for j in range(M) for l in range(M)).real
Grec = np.array([[1 if j == l else np.mean(I*np.conj(ph(j, l)))/0.25 for l in range(M)] for j in range(M)])
rec_err = float(np.abs(Grec - G0).max())
print("max reconstruction error:", rec_err, "| witness from image:", np.real(np.trace(P @ Grec))/4)

pairwise difference vectors distinct: True
max reconstruction error: 2.482534153247273e-16 | witness from image: 0.9999999999999998


## 3. Path-phase randomisation and figure
With probability $\epsilon$ the path phases are replaced by independent uniform phases: $G\mapsto(1-\epsilon)G+\epsilon I$.

In [5]:
eps = np.linspace(0, 0.08, 81)
coll = np.array([np.real(np.trace(P @ ((1-e)*G0 + e*np.eye(M))))/4 for e in eps])
eps_cross = 2*(1 - c_star)
print(f"collected fraction falls to c* at eps = {eps_cross:.4f}")
fig, ax = plt.subplots(1, 2, figsize=(7.2, 3.0))
im = ax[0].imshow(I, extent=[-np.pi, np.pi, -np.pi, np.pi], origin="lower", cmap="magma")
ax[0].set_xlabel(r"$k_x d$"); ax[0].set_ylabel(r"$k_y d$"); ax[0].set_title(r"(a) far field of $G_0$", fontsize=9)
fig.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].plot(100*eps, coll, "k-", label=r"marked state $G_0$")
ax[1].axhline(c_star, color="C3", ls="--", label="phase-only maximum")
ax[1].axvline(100*eps_cross, color="0.5", ls=":", lw=0.8)
ax[1].set_xlabel(r"path-phase randomization fraction $\epsilon$ (%)"); ax[1].set_ylabel(r"collected fraction $\mathrm{tr}(PG)/4$")
ax[1].set_title("(b) subspace witness", fontsize=9); ax[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig("results/fig_m4_witness.pdf"); fig.savefig("results/fig_m4_witness.png", dpi=200); plt.show()
json.dump(dict(jones_rows=[[str(x) for x in row] for row in V], collected_marked=1.0,
    phase_only_max_closed_form=float(c_star), phase_only_maximiser_over_pi=[0, 7/4, 15/8, 1/8],
    phase_only_upper_bound_bnb=float(cert_ub), bnb_tolerance=tol, margin=float(1-c_star),
    randomization_crossing=float(eps_cross), far_field_recovery_error=rec_err),
    open("results/four_path_results.json", "w"), indent=1)
print("saved results/four_path_results.json and results/fig_m4_witness.pdf")

collected fraction falls to c* at eps = 0.0347


saved results/four_path_results.json and results/fig_m4_witness.pdf


## 4. Exploratory survey, $M=2,\dots,5$
For random Hermitian $W$: maximum of $\mathrm{tr}(WG)$ over $\mathcal{E}_M$ (semidefinite program) versus multi-start phase-only optimisation. A positive difference is a candidate gap only (a local maximum is a lower bound).

In [6]:
def sdp(W, M):
    G = cp.Variable((M, M), hermitian=True)
    p = cp.Problem(cp.Maximize(cp.real(cp.trace(W @ G))), [G >> 0, cp.diag(G) == 1]); p.solve(solver="CLARABEL")
    return p.value, G.value
def phase_max(W, M, rng, R):
    fo = lambda t: -np.real(np.exp(-1j*np.r_[0, t]) @ W @ np.exp(1j*np.r_[0, t]))
    return max(-minimize(fo, rng.uniform(0, 2*np.pi, M-1)).fun for _ in range(R))
rng = np.random.default_rng(1); survey = []
for Mm, n in [(2, 150), (3, 150), (4, 150), (5, 80)]:
    gaps, ranks, t0 = [], set(), time.time()
    for i in range(n):
        A = rng.normal(size=(Mm, Mm)) + 1j*rng.normal(size=(Mm, Mm)); W = (A + A.conj().T)/2
        sv, G = sdp(W, Mm); pv = phase_max(W, Mm, rng, 60); gap = (sv - pv)/abs(sv); gaps.append(gap)
        if gap > 1e-4: ranks.add(int(np.sum(np.linalg.eigvalsh(G) > 1e-5*Mm)))
        if (i+1) % 25 == 0 or i+1 == n: print(f"M={Mm}: {i+1}/{n} trials, {time.time()-t0:.0f}s", flush=True)
    gaps = np.array(gaps)
    survey.append(dict(M=Mm, trials=n, max_rel_gap=float(gaps.max()), frac_gap=float(np.mean(gaps > 1e-4)), ranks=sorted(ranks)))
    print(survey[-1])
json.dump(survey, open("results/survey_results.json", "w"), indent=1); print("saved results/survey_results.json")

M=2: 150/150 trials, 13s
{'M': 2, 'trials': 150, 'max_rel_gap': -5.852566166822222e-11, 'frac_gap': 0.0, 'ranks': []}
M=3: 150/150 trials, 24s
{'M': 3, 'trials': 150, 'max_rel_gap': -2.271304394465483e-10, 'frac_gap': 0.0, 'ranks': []}
M=4: 150/150 trials, 34s
{'M': 4, 'trials': 150, 'max_rel_gap': 0.0249071983488348, 'frac_gap': 0.02, 'ranks': [2]}
M=5: 80/80 trials, 24s
{'M': 5, 'trials': 80, 'max_rel_gap': 0.017727708112113834, 'frac_gap': 0.1125, 'ranks': [2]}
saved results/survey_results.json
